# ElectPyNasa — Colab Pipeline Runner

This notebook turns Google Colab into a **remote execution environment** for the
ElectPyNasa CLI. It does **not** duplicate any pipeline logic — it simply:

1. **Clones** the GitHub repository `Vtheonly/Nasa_Hackathon_2025`
2. **Installs** every dependency declared by the project
3. **Runs** the exact same CLI you would run on a local machine:
   - `electpynasa.cli.grayscale` — GHS stretch for single FITS/TIFF channels
   - `electpynasa.cli.composite` — three-channel RGB composite with alignment
   - `electpynasa.cli.pyramid` — DZI pyramid generation via libvips
4. **Zips** the generated DZI pyramid into a single archive
5. **Saves** the archive to Google Drive (if mounted) and offers a browser download

> **Single source of truth.** The Colab notebook never re-implements any
> processing logic. Whatever runs here is the same code that runs on your
> laptop — Colab is just a beefier, faster-network environment.

---

### Workflow at a glance

```
┌───────────────────────────── Google Colab ─────────────────────────────┐
│                                                                        │
│  1. Clone repo      →   /content/Nasa_Hackathon_2025                   │
│  2. Install deps    →   pip + apt (libvips)                            │
│  3. Stage inputs    →   upload / Drive / sample URL                    │
│  4. Run CLI         →   python -m electpynasa.cli.<pipeline> ...       │
│  5. Zip output      →   pyramid → pyramid.zip                          │
│  6. Save / download →   Drive + browser download                       │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

## Step 1 — Configuration

Edit the values below to control the pipeline run. All paths are relative to
`/content/` (Colab's filesystem root).

In [ ]:
# ==================== USER CONFIGURATION ====================

# GitHub repository to clone
GITHUB_REPO_URL = "https://github.com/Vtheonly/Nasa_Hackathon_2025.git"
REPO_DIR_NAME   = "Nasa_Hackathon_2025"   # local directory name after clone

# Where to store inputs / outputs on Colab
WORK_DIR        = "/content/workspace"
INPUTS_DIR      = f"{WORK_DIR}/inputs"
OUTPUTS_DIR     = f"{WORK_DIR}/outputs"

# Where the DZI pyramid will be written (input to the pyramid CLI)
PYRAMID_OUTPUT_DIR = f"{OUTPUTS_DIR}/deepzoom"

# Final ZIP archive path
ZIP_PATH = f"{WORK_DIR}/electpynasa_pyramid.zip"

# Mount Google Drive? (set to False to skip — outputs stay on Colab only)
MOUNT_DRIVE = True
DRIVE_DEST  = "/content/drive/MyDrive/ElectPyNasa"  # where to copy the ZIP on Drive

# ==================== INTERNAL (do not edit) ====================
import os, sys, shutil, subprocess, json, time
from pathlib import Path

WORK_DIR = Path(WORK_DIR)
INPUTS_DIR = Path(INPUTS_DIR)
OUTPUTS_DIR = Path(OUTPUTS_DIR)
PYRAMID_OUTPUT_DIR = Path(PYRAMID_OUTPUT_DIR)
ZIP_PATH = Path(ZIP_PATH)

WORK_DIR.mkdir(parents=True, exist_ok=True)
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace:  {WORK_DIR}")
print(f"Inputs:     {INPUTS_DIR}")
print(f"Outputs:    {OUTPUTS_DIR}")
print(f"Pyramid:    {PYRAMID_OUTPUT_DIR}")
print(f"ZIP:        {ZIP_PATH}")

## Step 2 — Mount Google Drive (Optional)

Mounting Drive gives you **persistent storage** — when Colab recycles the
runtime, anything in `/content/` is lost, but anything in
`/content/drive/MyDrive/` survives. Set `MOUNT_DRIVE = False` in Step 1 to
skip.

In [ ]:
DRIVE_MOUNTED = False
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        if Path("/content/drive/MyDrive").exists():
            DRIVE_MOUNTED = True
            Path(DRIVE_DEST).mkdir(parents=True, exist_ok=True)
            print(f"[drive] Mounted. Persistent destination: {DRIVE_DEST}")
        else:
            print("[drive] Mount ran but MyDrive not visible — continuing without Drive.")
    except Exception as exc:
        print(f"[drive] Mount failed: {exc} — continuing without Drive.")
else:
    print("[drive] MOUNT_DRIVE=False — outputs stay on Colab only.")

## Step 3 — Clone the GitHub Repository

We clone into `/content/` so the project lives next to the workspace. If the
repo was already cloned (e.g. you re-ran the notebook), we pull the latest
changes instead of re-cloning.

In [ ]:
REPO_PATH = Path("/content") / REPO_DIR_NAME

def run(cmd, **kw):
    """Run a shell command, streaming output to the notebook."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result

if REPO_PATH.exists():
    print(f"[repo] Already cloned at {REPO_PATH} — pulling latest...")
    run(f"cd {REPO_PATH} && git pull --ff-only")
else:
    print(f"[repo] Cloning {GITHUB_REPO_URL} → {REPO_PATH}")
    run(f"cd /content && git clone --depth 1 {GITHUB_REPO_URL} {REPO_DIR_NAME}")

# Verify the structure
electpynasa_dir = REPO_PATH / "electpynasa"
galaxyviewer_dir = REPO_PATH / "galaxyviewer"
assert electpynasa_dir.exists(), f"Expected electpynasa/ at {electpynasa_dir}"
print(f"\n[repo] Structure verified:")
print(f"  - electpynasa/   : {electpynasa_dir}")
print(f"  - galaxyviewer/  : {galaxyviewer_dir} ({'present' if galaxyviewer_dir.exists() else 'absent'})")
print(f"  - colab/         : {REPO_PATH / 'colab'}")

## Step 4 — Install Project Dependencies

The project declares its Python dependencies in `electpynasa/requirements.txt`.
We also install **libvips** via `apt` because the pyramid CLI shells out to
the `vips dzsave` command.

In [ ]:
# 4a. Install libvips (system dependency for the pyramid CLI)
print("[deps] Installing libvips...")
run("apt-get update -qq && apt-get install -y -qq libvips-tools 2>&1 | tail -3")
print("[deps] Verifying vips CLI:")
run("vips --version")

# 4b. Install Python dependencies from the project's requirements.txt
req_path = electpynasa_dir / "requirements.txt"
print(f"\n[deps] Installing Python dependencies from {req_path}")
run(f"pip install -q -r {req_path}")
print("[deps] Python dependencies installed.")

# 4c. Install the electpynasa package in editable mode (provides CLI entry points)
print(f"\n[deps] Installing electpynasa package (editable)...")
run(f"pip install -q -e {electpynasa_dir}")
print("[deps] electpynasa installed.")

# 4d. Verify the CLI is on PATH
print("\n[deps] Verifying CLI entry points:")
run("which electpynasa-grayscale electpynasa-composite electpynasa-pyramid || true")

## Step 5 — Stage Input Images

You have three options for getting input images into the Colab runtime:

| Option | Best for | How |
|--------|----------|-----|
| **A. Upload** | A few small files you have locally | Use the Colab file browser (left sidebar → Files → Upload) |
| **B. Drive**  | Files already on your Drive | Mount Drive (Step 2) and copy them into `/content/workspace/inputs/` |
| **C. URL**    | Public files (e.g. MAST direct URLs) | Use the helper below |

The cell below provides a `download_input(url, filename)` helper and downloads
a small **sample FITS** file so you can immediately test the pipeline even
without uploading anything.

In [ ]:
import urllib.request

def download_input(url: str, filename: str | None = None) -> Path:
    """Download a file into the inputs directory with a progress hook."""
    filename = filename or url.rsplit("/", 1)[-1].split("?")[0] or "input.bin"
    dest = INPUTS_DIR / filename
    print(f"[input] Downloading {url} → {dest}")

    def progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = min(100, downloaded * 100 // total_size)
            bar = "=" * (pct // 2) + ">" + " " * (50 - pct // 2)
            print(f"\r[input] [{bar}] {pct:3d}%  ({downloaded // 1024} KiB)", end="", flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=progress)
    print(f"\n[input] Done: {dest} ({dest.stat().st_size:,} bytes)")
    return dest

# === OPTION C: Download a sample input ===
# Replace this URL with any FITS/TIFF you want to process.
# (The example below uses a small public FITS from the astropy test suite.)
SAMPLE_URL = "https://fits.gsfc.nasa.gov/samples/WFPC2u5780205r_c0fx.fits"
SAMPLE_NAME = "wfpc2_sample.fits"

sample_path = download_input(SAMPLE_URL, SAMPLE_NAME)
print(f"\n[input] Inputs directory contents:")
for p in sorted(INPUTS_DIR.glob("*")):
    print(f"  - {p.name:40s}  ({p.stat().st_size / 1024:.1f} KiB)")

# To upload your own files, use the Colab file browser sidebar and
# drag them into /content/workspace/inputs/.

## Step 6 — Run the Grayscale GHS Pipeline

Runs the project's existing `electpynasa.cli.grayscale` entry point on a
single FITS/TIFF channel. All CLI options are exposed below — defaults match
the project's `GHSConfig` recommended values.

> **Single source of truth.** This cell calls the exact same code path you
> would call from your laptop. No Colab-specific logic here.

In [ ]:
# ==================== GRAYSCALE CONFIG ====================
# Pick the input file you want to stretch (defaults to the sample downloaded above).
GRAYSCALE_INPUT = str(sample_path)   # change to e.g. str(INPUTS_DIR / "my_image.fits")
GRAYSCALE_OUTPUT_BASE = str(OUTPUTS_DIR / Path(GRAYSCALE_INPUT).stem + "_ghs")

# GHS parameters — these mirror `electpynasa.config.GHSConfig` defaults.
K = 2.5       # stretch factor
L = 6.0       # local decay
S = 0.20      # symmetry point
SP = 0.01     # shadow protection
HP = 0.98     # highlight protection

# ==================== RUN CLI ====================
cmd = (
    f"python -m electpynasa.cli.grayscale "
    f"--input {GRAYSCALE_INPUT!r} "
    f"--output {GRAYSCALE_OUTPUT_BASE!r} "
    f"--k {K} --L {L} --s {S} --sp {SP} --hp {HP}"
)
print(f"[grayscale] Running: {cmd}\n")
run(cmd)

# Locate the output file
grayscale_out = Path(f"{GRAYSCALE_OUTPUT_BASE}_grayscale.tif")
assert grayscale_out.exists(), f"Expected output not found: {grayscale_out}"
print(f"\n[grayscale] Output: {grayscale_out} ({grayscale_out.stat().st_size / 1024:.1f} KiB)")

## Step 7 — Run the Color Composite Pipeline (Optional)

If you have **three** FITS/TIFF channels (red / green / blue), this cell runs
`electpynasa.cli.composite` to align them, balance colors, and produce both a
32-bit HDR master and an 8-bit display preview.

> Skip this cell if you only have a single channel — the pyramid pipeline
> (Step 8) works on any TIFF, including the grayscale output from Step 6.

In [ ]:
# ==================== COMPOSITE CONFIG ====================
# Provide paths to three channel files. Leave as None to skip this step.
RED_INPUT   = None   # e.g. str(INPUTS_DIR / "F444W.fits")
GREEN_INPUT = None   # e.g. str(INPUTS_DIR / "F200W.fits")
BLUE_INPUT  = None   # e.g. str(INPUTS_DIR / "F090W.fits")

# Output base
COMPOSITE_OUTPUT_BASE = str(OUTPUTS_DIR / "composite_result")

# Tuning (mirror `electpynasa.config.CompositePipelineConfig` defaults)
SATURATION = 1.35
Q_FACTOR   = 7.5

# ==================== RUN CLI (or skip) ====================
if all([RED_INPUT, GREEN_INPUT, BLUE_INPUT]):
    cmd = (
        f"python -m electpynasa.cli.composite "
        f"--r {RED_INPUT!r} --g {GREEN_INPUT!r} --b {BLUE_INPUT!r} "
        f"--output {COMPOSITE_OUTPUT_BASE!r} "
        f"--saturation {SATURATION} --q {Q_FACTOR}"
    )
    print(f"[composite] Running: {cmd}\n")
    run(cmd)

    composite_hdr = Path(f"{COMPOSITE_OUTPUT_BASE}_color_32bit.tiff")
    composite_preview = Path(f"{COMPOSITE_OUTPUT_BASE}_color_8bit_preview.tiff")
    print(f"\n[composite] HDR master: {composite_hdr} ({composite_hdr.stat().st_size / 1024:.1f} KiB)")
    print(f"[composite] Preview:    {composite_preview} ({composite_preview.stat().st_size / 1024:.1f} KiB)")
else:
    print("[composite] Skipping — set RED_INPUT / GREEN_INPUT / BLUE_INPUT to enable.")
    composite_preview = None

## Step 8 — Generate the DZI Pyramid

Runs `electpynasa.cli.pyramid` on the chosen input image. The pipeline:

1. Validates the input file
2. Verifies the libvips CLI is available
3. Prepares the output directory (`<basename>/<basename>_files/`)
4. Invokes `vips dzsave` with the configured tile size, overlap, format, and quality
5. Emits a `<basename>.dzi` XML manifest

The output is a hierarchical directory tree of tiles — exactly the format
**GalaxyViewer** consumes.

In [ ]:
# ==================== PYRAMID CONFIG ====================
# Choose the input for pyramid generation. Defaults to the grayscale output
# from Step 6, but you can point it at any TIFF/JPEG/PNG/WebP.
PYRAMID_INPUT = str(grayscale_out) if grayscale_out.exists() else str(sample_path)

# Pyramid parameters (mirror `electpynasa.config.PyramidConfig` defaults)
TILE_SIZE = 256
OVERLAP   = 1
TILE_FORMAT = "jpeg"   # jpeg | png | webp
QUALITY   = 90

# ==================== RUN CLI ====================
cmd = (
    f"python -m electpynasa.cli.pyramid "
    f"--input {PYRAMID_INPUT!r} "
    f"--output {PYRAMID_OUTPUT_DIR!r} "
    f"--tileSize {TILE_SIZE} --overlap {OVERLAP} "
    f"--format {TILE_FORMAT} --quality {QUALITY}"
)
print(f"[pyramid] Running: {cmd}\n")
run(cmd)

# Locate the generated .dzi manifest
dzi_files = list(PYRAMID_OUTPUT_DIR.rglob("*.dzi"))
assert dzi_files, f"No .dzi manifest generated under {PYRAMID_OUTPUT_DIR}"
dzi_path = dzi_files[0]
print(f"\n[pyramid] DZI manifest: {dzi_path}")
print(f"[pyramid] Tiles directory: {dzi_path.parent}")

# Show the manifest contents
print(f"\n[pyramid] Manifest XML:")
print(dzi_path.read_text())

## Step 9 — Compress the Pyramid into a ZIP Archive

The DZI pyramid is a directory tree (one `.dzi` XML file + one
`<basename>_files/` folder containing hundreds of tile JPEGs). We zip the
whole tree into a single archive so it can be downloaded or stored on Drive
in one shot.

In [ ]:
print(f"[zip] Compressing {dzi_path.parent} → {ZIP_PATH}")

# Remove any previous archive
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Use Python's shutil.make_archive for cross-platform reliability.
# We zip the *contents* of the dzi_path.parent directory (not the parent
# folder itself) so the archive root contains the .dzi file directly.
import zipfile
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    pyramid_root = dzi_path.parent
    for file_path in pyramid_root.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(pyramid_root)
            zf.write(file_path, arcname)
            print(f"  + {arcname}")

print(f"\n[zip] Archive complete: {ZIP_PATH}")
print(f"[zip] Size: {ZIP_PATH.stat().st_size / 1024 / 1024:.2f} MiB")
print(f"[zip] File count: {len(list(zipfile.ZipFile(ZIP_PATH).namelist()))}")

## Step 10 — Save the ZIP to Drive + Browser Download

Two delivery paths, both run if available:

- **Google Drive** — copies the ZIP to `MyDrive/ElectPyNasa/` (if Drive is mounted)
- **Browser download** — uses Colab's `files.download()` so the file lands in
  your browser's download folder

In [ ]:
# 10a. Copy to Google Drive
if DRIVE_MOUNTED:
    drive_zip = Path(DRIVE_DEST) / ZIP_PATH.name
    print(f"[drive] Copying {ZIP_PATH} → {drive_zip}")
    shutil.copy2(ZIP_PATH, drive_zip)
    print(f"[drive] Done. Drive path: {drive_zip}")
else:
    print("[drive] Drive not mounted — skipping Drive copy.")

# 10b. Trigger a browser download
try:
    from google.colab import files
    print(f"\n[download] Triggering browser download of {ZIP_PATH.name}...")
    files.download(str(ZIP_PATH))
except ImportError:
    print("[download] Not running on Colab — copy the file manually from the file browser.")
except Exception as exc:
    print(f"[download] Could not trigger browser download: {exc}")
    print(f"           The file is still available at: {ZIP_PATH}")

## Step 11 — Inspect the Output

Quick utility cells to list what was generated and preview the first tile.

In [ ]:
# List the workspace contents
print("=== Workspace tree (top 2 levels) ===")
def tree(root: Path, prefix: str = "", depth: int = 0, max_depth: int = 2):
    if depth > max_depth: return
    children = sorted([p for p in root.iterdir() if not p.name.startswith(".")])
    for i, p in enumerate(children):
        connector = "└── " if i == len(children) - 1 else "├── "
        size_str = f"  ({p.stat().st_size / 1024:.1f} KiB)" if p.is_file() else ""
        print(f"{prefix}{connector}{p.name}{size_str}")
        if p.is_dir():
            extension = "    " if i == len(children) - 1 else "│   "
            tree(p, prefix + extension, depth + 1, max_depth)

tree(WORK_DIR)

In [ ]:
# Preview a sample tile from the pyramid
import matplotlib.pyplot as plt
from PIL import Image

tiles = list(dzi_path.parent.rglob("*.jpg"))
if not tiles:
    tiles = list(dzi_path.parent.rglob("*.png"))
if tiles:
    sample = tiles[0]
    print(f"Previewing first tile: {sample.name}")
    img = Image.open(sample)
    print(f"  Size: {img.size}, Mode: {img.mode}")

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(sample.name)
    plt.axis("off")
    plt.show()

    print(f"\nTotal tiles generated: {len(tiles)}")
else:
    print("No tiles found.")

## Step 12 — View with GalaxyViewer (Local)

The repo also includes **GalaxyViewer**, a modern DZI viewer. To use it
locally after downloading the ZIP:

```bash
# Unzip the pyramid
unzip electpynasa_pyramid.zip -d pyramid/

# Clone the repo and run GalaxyViewer
git clone https://github.com/Vtheonly/Nasa_Hackathon_2025.git
cd Nasa_Hackathon_2025/galaxyviewer
npm install
npm run dev

# In another terminal, serve the pyramid files:
cd path/to/pyramid
python3 -m http.server 8000

# Then open the viewer:
# http://localhost:5173/?src=http://localhost:8000/<basename>.dzi
```

GalaxyViewer features progressive tile loading, smooth zoom/pan, a minimap,
keyboard shortcuts, and a polished dark UI.

## Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `git clone` fails | Repo is private or URL is wrong | Set `GITHUB_REPO_URL` to the correct HTTPS URL |
| `vips: command not found` | apt install failed | Re-run Step 4; if it persists, run `!apt-get install -y libvips-tools` manually |
| `No module named 'electpynasa'` | Package install failed | Run `!pip install -e /content/Nasa_Hackathon_2025/electpynasa` manually |
| Grayscale pipeline fails on the sample | Sample URL is unreachable | Set `GRAYSCALE_INPUT` to a local file you uploaded via the Colab file browser |
| Pyramid CLI emits `Pipeline step 'check_vips' failed` | libvips missing | Re-run Step 4 |
| Browser download doesn't start | Pop-up blocked | Allow pop-ups for Colab; the file is still at `ZIP_PATH` |
| Drive copy fails with `Drive not mounted` | Step 2 skipped | Set `MOUNT_DRIVE = True` and re-run Step 2 |
| ZIP is huge (>1 GiB) | Image is very large + JPEG quality 90 | Lower `QUALITY` to 75, or use `TILE_FORMAT = "webp"` |

---

### Output layout

After a successful run, your workspace looks like:

```
/content/workspace/
├── inputs/
│   └── wfpc2_sample.fits                 ← downloaded sample
├── outputs/
│   ├── wfpc2_sample_grayscale.tif        ← GHS-stretched grayscale
│   └── deepzoom/
│       └── wfpc2_sample_grayscale/
│           ├── wfpc2_sample_grayscale.dzi   ← DZI manifest
│           └── wfpc2_sample_grayscale_files/
│               ├── 0/                       ← pyramid level 0 (smallest)
│               │   └── 0_0.jpg
│               ├── 8/                       ← pyramid level 8 (largest)
│               │   ├── 0_0.jpg
│               │   ├── 0_1.jpg
│               │   └── ...
│               └── ...
└── electpynasa_pyramid.zip              ← final archive (downloaded + on Drive)
```